# Data Preparation: Labour Well-being in Higher Education (AT & CZ)

**Project Goal:** Investigate the determinants of labor well-being (focusing on *Burnout*) among academic staff in Austria and the Czech Republic using survey data from 2024.

**Notebook Objective:** Load the survey data, perform necessary cleaning, calculate composite scores from Likert scale items, handle missing values and outliers, and save a processed dataset ready for Exploratory Data Analysis (EDA) and modeling.

**Input Data:** The primary input is an Excel file containing survey responses.

**Main Steps:**
1. Load and Initial Inspection of the Dataset.
2. Definition of Survey Items for Composite Scales.
3. Scale Inversion (where necessary).
4. Evaluation of Likert Scale Reliability (Critical Step).
5. Calculation of Composite Variables (Averages).
6. Dataset Cleanup (Combine columns, drop originals, rename).
7. Handling Missing Values (NaNs) and Outliers.
8. Saving the Processed Dataset.

In [1]:
# Core Libraries
import pandas as pd
import numpy as np
import warnings # To manage warnings

# Data Visualization (Recommended for NaN analysis)
# import missingno as msno # Uncomment if you want to visualize missing data
# import matplotlib.pyplot as plt # Uncomment for plotting
# import seaborn as sns # Uncomment for plotting

# Machine Learning Libraries (for Imputation/Scaling)
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Statistical Libraries (Recommended for Reliability Analysis)
# import pingouin as pg # Uncomment if you want to calculate Cronbach's Alpha

# Settings
pd.set_option('display.max_columns', 100) # Show more columns
pd.set_option('display.max_rows', 100) # Show more rows
warnings.filterwarnings('ignore', category=FutureWarning) # Ignore specific warnings if needed

## 1. Load Data and Initial Inspection

Load the dataset from the specified path. It's recommended to use relative paths or environment variables for better portability. We'll perform a basic inspection (`.head()`, `.info()`, `.describe()`) to understand the initial structure.

In [2]:
# --- Configuration ---
# !! IMPORTANT: Update this path to your actual file location !!
# Using a raw string literal (r"...") or forward slashes is safer for paths.
# Consider making this path a parameter or reading from a config file.
# file_path = r'/content/drive/MyDrive/labour well being/Data preparation /dataset_vulnerability scores_after imputation.xlsx'
# Using a placeholder for demonstration if Google Drive isn't mounted:
file_path = 'dataset_vulnerability_scores_after_imputation.xlsx' # Placeholder

# --- Load Data ---
try:
    df = pd.read_excel(file_path)
    print(f"Successfully loaded data from: {file_path}")
    print(f"Initial DataFrame shape: {df.shape}")
except FileNotFoundError:
    print(f"ERROR: File not found at {file_path}. Please check the path.")
    # Create an empty DataFrame to avoid errors in subsequent cells if needed for testing
    # df = pd.DataFrame()
    # Or exit the script:
    # import sys
    # sys.exit("Exiting due to file not found.")
except Exception as e:
    print(f"An error occurred while loading the Excel file: {e}")
    # df = pd.DataFrame() # Or exit
    # import sys
    # sys.exit("Exiting due to file loading error.")

# --- Initial Inspection (only if df was loaded successfully) ---
if 'df' in locals() and not df.empty:
    print("\n--- First 5 Rows ---")
    display(df.head()) # Use display() in environments like Jupyter/Colab

    print("\n--- DataFrame Info ---")
    df.info()

    print("\n--- Descriptive Statistics (Numeric Columns) ---")
    display(df.describe())

    print("\n--- Descriptive Statistics (Object Columns) ---")
    display(df.describe(include='object'))
else:
    print("\nSkipping initial inspection as DataFrame could not be loaded.")

Successfully loaded data from: dataset_vulnerability_scores_after_imputation.xlsx
Initial DataFrame shape: (2708, 132)

--- First 5 Rows ---


,Country,Version,1. Gender:,2. Age (in years):,Unnamed: 4,3. Nationality:,4. Current marital (partnership) status:,5. Do you currently care for underage children or dependent relatives?,6. The type of higher education insitution where you primarily work:,7. Subject area of the faculty (higher education institution) where you primarily work:,8. Duration of your current employment contract at the higher education institution where you primarily work:,"9. Extent of employment in higher education (in hours/week, aggregated for all higher education institutions where you work):",10. Actual average weekly working hours in higher education (in a typical semester week):,"Effort (less, more, equal)",Effort [%],Income CZK,Income EURO,Euro Adj.,Salary/hour,Salary effort/hour,12. Do you hold a leadership position at a higher education institution?,13. How influential are you in helping to shape key academic policies at your institution at the level of department or similar unit?,14. Do you currently have another (paid) job outside higher education?,14.1. Actual average weekly working hours outside higher education (in a typical semester week):,14.1. Actual average weekly working hours outside higher education (in a typical semester week): .1,Academic/Non-academic,"CZ_15. Your current position at the higher education institution, where you primarily work:","AT_15. Your current position at the higher education institution, where you primarily work:",16. Please choose the category that best fits your job description:,17. The highest level of education attained:,18. Total length of your career in Czech higher education in years:,"1. Teaching (classroom instruction, preparation of instructional materials and lesson plans, advising students, reading and evaluating student work, examination management, etc.)","2. Research (reading literature, designing and conducting experiments, collecting and analysing data, writing articles or other scientific texts, etc.)","3. Activities related to externally funded research projects (searching for information on available funding sources, preparation of grant applications and project reports, project management and administration, etc.)","4. Organisational and administrative activities (organising and attending meetings, dealing with tasks and documents not directly related to teaching, research, or externally funded research projects, etc.)",Teaching %,Research %,Activities related to externally funded research projects %,Organisational and administrative activities %,"1. Facilities and technological equipment (offices, classrooms, laboratories, computers, projectors, teaching software)","2. Research equipment (instruments, tools, materials, software for working with data)",3. Availability of scientific literature (access to up-to-date scientific articles and books),"4. Personnel support (support from secretarial, administrative and technical staff, laboratory technicians, etc.)","5. Opportunities for personal and professional development (offer and availability of professional courses and internships, opportunities to participate at conferences and other professional events)",6. Availability of resources for research funding,"1. At my institution, there is a strong performance orientation.",2. I feel like I can be myself at my job.,"3. At work, I often feel like I have to follow other people’s commands.","4. If I could choose, I would do things at work differently.",5. The tasks I have to do at work are in line with what I really want to do.,...,19. Because otherwise I will feel bad about myself.,1. How often have you felt worn out?,2. How often have you been physically exhausted?,3. How often have you been emotionally exhausted?,4. How often have you felt tired?,"1. For me, work is the most important part of life.","2. As far as my professional development is concerned, I consider myself to be fairly ambitious.","3. If necessary, I work until exhaustion.",4. My work should always be faultless.,5. I conti


--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2708 entries, 0 to 2707
Columns: 132 entries, Country to Vulnerability
dtypes: float64(80), int64(48), object(4)
memory usage: 2.7+ MB

--- Descriptive Statistics (Numeric Columns) ---


,Country,Version,1. Gender:,2. Age (in years):,Unnamed: 4,4. Current marital (partnership) status:,5. Do you currently care for underage children or dependent relatives?,6. The type of higher education insitution where you primarily work:,7. Subject area of the faculty (higher education institution) where you primarily work:,8. Duration of your current employment contract at the higher education institution where you primarily work:,10. Actual average weekly working hours in higher education (in a typical semester week):,"Effort (less, more, equal)",Effort [%],Income CZK,Income EURO,Euro Adj.,Salary/hour,Salary effort/hour,12. Do you hold a leadership position at a higher education institution?,13. How influential are you in helping to shape key academic policies at your institution at the level of department or similar unit?,14. Do you currently have another (paid) job outside higher education?,14.1. Actual average weekly working hours outside higher education (in a typical semester week):,14.1. Actual average weekly working hours outside higher education (in a typical semester week): .1,Academic/Non-academic,"CZ_15. Your current position at the higher education institution, where you primarily work:","AT_15. Your current position at the higher education institution, where you primarily work:",16. Please choose the category that best fits your job description:,17. The highest level of education attained:,"1. Teaching (classroom instruction, preparation of instructional materials and lesson plans, advising students, reading and evaluating student work, examination management, etc.)","2. Research (reading literature, designing and conducting experiments, collecting and analysing data, writing articles or other scientific texts, etc.)","3. Activities related to externally funded research projects (searching for information on available funding sources, preparation of grant applications and project reports, project management and administration, etc.)",Teaching %,Research %,Activities related to externally funded research projects %,Organisational and administrative activities %,"1. Facilities and technological equipment (offices, classrooms, laboratories, computers, projectors, teaching software)","2. Research equipment (instruments, tools, materials, software for working with data)",3. Availability of scientific literature (access to up-to-date scientific articles and books),"4. Personnel support (support from secretarial, administrative and technical staff, laboratory technicians, etc.)","5. Opportunities for personal and professional development (offer and availability of professional courses and internships, opportunities to participate at conferences and other professional events)",6. Availability of resources for research funding,"1. At my institution, there is a strong performance orientation.",2. I feel like I can be myself at my job.,"3. At work, I often feel like I have to follow other people’s commands.","4. If I could choose, I would do things at work differently.",5. The tasks I have to do at work are in line with what I really want to do.,6. I feel free to do my job the way I think it could best be done.,"7. In my job, I feel forced to do things I do not want to do.",1. ... makes sure that the members of staff have good development opportunities.,2. ... gives high priority to job satisfaction.,...,19. Because otherwise I will feel bad about myself.,1. How often have you felt worn out?,2. How often have you been physically exhausted?,3. How often have you been emotionally exhausted?,4. How often have you felt tired?,"1. For me, work is the most important part of life.","2. As far as my professional development is concerned, I consider myself to be fairly ambitious.","3. If necessary, I work until exhaustion.",4. My work should always be faultless.,5. I continue thinking about work problems during my leisure time.,"6. If I am not successful, I give up quickly.",7. Lack of success can give me new determination.,8. I d


--- Descriptive Statistics (Object Columns) ---


,3. Nationality:,"9. Extent of employment in higher education (in hours/week, aggregated for all higher education institutions where you work):",18. Total length of your career in Czech higher education in years:,"4. Organisational and administrative activities (organising and attending meetings, dealing with tasks and documents not directly related to teaching, research, or externally funded research projects, etc.)"
count,266,2674,536,2055
unique,118,67,53,62
top,Slovenská republika,40,5,5
freq,39,1821,46,367


## 2. Define Column Lists for Composite Variable Calculation

These lists contain the exact column names from the survey data corresponding to the items used to calculate composite scores (e.g., Academic Resources, Motivation sub-scales). Separate lists identify items needing scale inversion.

*Self-Correction:* Checked for potential trailing spaces or inconsistencies in names based on original code.

In [3]:
# --- Define Column Groups for Composite Variables ---

# Work Conditions / Environment
cols_academic_resources = [
    '1. Facilities and technological equipment (offices, classrooms, laboratories, computers, projectors, teaching software)',
    '2. Research equipment (instruments, tools, materials, software for working with data)',
    '3. Availability of scientific literature (access to up-to-date scientific articles and books)',
    '4. Personnel support (support from secretarial, administrative and technical staff, laboratory technicians, etc.)',
    '5. Opportunities for personal and professional development (offer and availability of professional courses and internships, opportunities to participate at conferences and other professional events)',
    '6. Availability of resources for research funding'
]
col_performance_pressure = '1. At my institution, there is a strong performance orientation.'
cols_perceived_autonomy = [
    '2. I feel like I can be myself at my job.',
    '3. At work, I often feel like I have to follow other people’s commands.', # Inverted
    '4. If I could choose, I would do things at work differently.',             # Inverted
    '5. The tasks I have to do at work are in line with what I really want to do.',
    '6. I feel free to do my job the way I think it could best be done.',
    '7. In my job, I feel forced to do things I do not want to do. '           # Inverted (potential trailing space corrected)
]
cols_autonomy_to_invert = [
    '3. At work, I often feel like I have to follow other people’s commands.',
    '4. If I could choose, I would do things at work differently.',
    '7. In my job, I feel forced to do things I do not want to do. ' # Keep potential space if it exists in data
]
cols_quality_leadership = [
    '1. ... makes sure that the members of staff have good development opportunities.',
    '2. ... gives high priority to job satisfaction.',
    '3. ... is good at work planning.',
    '4. ... is good at solving conflicts. ' # Potential trailing space
]
cols_sense_community = [
    '1. There is a good atmosphere between myself and my colleagues. ', # Potential trailing space
    '2. There is good co-operation between the colleagues at work.', # Corrected - no trailing space in original comment
    '3. I feel part of a community at my place of work.' # Corrected - no trailing space in original comment
]
cols_job_satisfaction = [
    '1. ... your work prospects? ', # Potential trailing space
    '2. ...  the physical working conditions (e.g. facilities, equipment, physical working environment)?', # Double space after ellipsis
    '3. ... the way your abilities are used?',
    '4. ... your salary?',
    '5. ... your job as a whole, everything taken into consideration? ' # Potential trailing space
]

# Work Motivation Sub-scales (WEIMS)
cols_amotivation = [
    '1. I don’t, because I really feel that I’m wasting my time at work.',
    '7. I do little because I don’t think this work is worth putting efforts into.',
    '13. I don’t know why I’m doing this job, it’s pointless work.'
]
cols_extrinsic_social = [
    '2. To get others’ approval (e.g., supervisor, colleagues, family, students...).',
    '8. Because others will respect me more (e.g., supervisor, colleagues, family, students...).',
    '14. To avoid being criticized by others (e.g., supervisor, colleagues, family, students...). ' # Potential trailing space
]
cols_extrinsic_material = [
    '3. Because others will reward me financially only if I put enough effort in my job (e.g., employer, supervisor...).',
    '9. Because others offer me greater job security if I put enough effort in my job (e.g., employer, supervisor…).',
    '15. Because I risk losing my job if I don’t put enough effort in it.'
]
cols_introjected = [
    '4. Because I have to prove to myself that I can.',
    '10. Because it makes me feel proud of myself.',
    '16. Because otherwise I will feel ashamed of myself.',
    '19. Because otherwise I will feel bad about myself.'
]
cols_identified = [
    '5. Because I personally consider it important to put efforts in this job.',
    '11. Because putting efforts in this job aligns with my personal values.',
    '17. Because putting efforts in this job has personal significance to me.'
]
cols_intrinsic = [
    '6. Because I have fun doing my job.',
    '12. Because what I do in my work is exciting.',
    '18. Because the work I do is interesting.'
]

# Burnout (CBI - Copenhagen Burnout Inventory - Personal Burnout subscale likely)
cols_burnout = [
    '1. How often have you felt worn out?',
    '2. How often have you been physically exhausted?',
    '3. How often have you been emotionally exhausted?',
    '4. How often have you felt tired?'
]

# Attitudes / Vulnerability to Burnout Sub-scales (AVEM)
cols_subjective_significance = [
    '1. For me, work is the most important part of life. ', # Potential trailing space
    '12. Work is everything to me. ', # Potential trailing space
    '23. I need work like the air I breathe. ', # Potential trailing space
    '34. I wouldn’t know how to live without work. ' # Potential trailing space
]
cols_professional_ambition = [
    '2. As far as my professional development is concerned, I consider myself to be fairly ambitious. ', # Potential trailing space
    '13. I aspire to higher career goals than most other people.',
    '24. I have great plans for my career. ', # Potential trailing space
    '35. Professional success is an important aim in my life. ' # Potential trailing space
]
cols_tendency_exertion = [
    '3. If necessary, I work until exhaustion.',
    '14. I don’t go easy on myself at work.',
    '25. I probably work more than I should. ', # Potential trailing space
    '36. I tend to overwork.'
]
cols_striving_perfection = [
    '4. My work should always be faultless.',
    '15. I prefer to check everything three times over rather than hand in work that contains mistakes. ', # Potential trailing space
    '26. Whatever I do, it must be perfect. ', # Potential trailing space
    '37. I don’t consider my work to be finished until I am completely satisfied with the result. ' # Potential trailing space
]
cols_emotional_distancing = [
    '5. I continue thinking about work problems during my leisure time.', # Inverted
    '16. After work I can switch off easily.  ', # Ordinal - Potential DOUBLE SPACE
    '27. Problems at work actually occupy my mind all day.', # Inverted
    '38. My thoughts revolve around work almost exclusively.' # Inverted
]
cols_distancing_to_invert = [
    '5. I continue thinking about work problems during my leisure time.',
    '27. Problems at work actually occupy my mind all day.',
    '38. My thoughts revolve around work almost exclusively.'
]
cols_resignation_tendencies = [
    '6. If I am not successful, I give up quickly.',
    '17. I find it difficult to cope with lack of success.',
    '28. Failures at work can easily discourage me.',
    '39. When I’m unsuccessful at work, it makes me feel very down.'
]
cols_offensive_coping = [
    '7. Lack of success can give me new determination. ', # Potential trailing space
    '18. If I don’t succeed at something, I remain persistent and try even harder.',
    '29. Lack of success doesn’t bring me down, but makes me to try even harder.',
    '40. If I don’t succeed at something, that just makes me all the more determined.'
]
cols_mental_stability = [
    '8. I don’t get upset easily. ', # Ordinal, potential trailing space
    '19. I consider myself to be rather scatterbrained.', # Inverted
    '30. The hustle and bustle around me doesn’t affect me.', # Ordinal
    '41. I can be calm and collected in almost all situations. ' # Ordinal, potential trailing space
]
cols_stability_to_invert = [
    '19. I consider myself to be rather scatterbrained.'
]
cols_satisfaction_work = [
    '9. So far, my professional life has been fairly successful.', # Corrected - no trailing space in original comment
    '20. Up to this point in my career, I have experienced more successes than disappointments. ', # Potential trailing space
    '31. So far, I have succeeded at almost everything in my professional development.  ', # Potential DOUBLE SPACE
    '42. My life up till now has been characterized by professional success. ' # Potential trailing space
]
cols_satisfaction_life = [
    '10. I have every reason to be optimistic about my future.',
    '21. I cannot complain about my life in any way.',
    '32. So far, I can be satisfied with my life. ', # Potential trailing space
    '43. By and large, I am happy and content. ' # Potential trailing space
]
cols_social_support = [
    '11. My partner (or my closest person) is understanding about my work.', # Ordinal
    '22. My family isn’t very interested in my problems at work.', # Inverted
    '33. I would like my partner (or my closest person) to have more consideration for my work duties and challenges.', # Inverted
    '44. I have the full support of my family. ' # Ordinal, potential trailing space
]
cols_support_to_invert = [
    '22. My family isn’t very interested in my problems at work.',
    '33. I would like my partner (or my closest person) to have more consideration for my work duties and challenges.'
]

# --- Collect all original survey item columns used in calculations ---
all_original_calc_cols = list(set(
    cols_academic_resources +
    [col_performance_pressure] +
    cols_perceived_autonomy +
    cols_quality_leadership +
    cols_sense_community +
    cols_job_satisfaction +
    cols_amotivation + cols_extrinsic_social + cols_extrinsic_material +
    cols_introjected + cols_identified + cols_intrinsic +
    cols_burnout +
    cols_subjective_significance + cols_professional_ambition + cols_tendency_exertion +
    cols_striving_perfection + cols_emotional_distancing + cols_resignation_tendencies +
    cols_offensive_coping + cols_mental_stability + cols_satisfaction_work +
    cols_satisfaction_life + cols_social_support
))

# --- Verify which columns actually exist in the DataFrame ---
if 'df' in locals() and not df.empty:
    existing_original_calc_cols = [col for col in all_original_calc_cols if col in df.columns]
    missing_original_calc_cols = [col for col in all_original_calc_cols if col not in df.columns]
    print(f"Found {len(existing_original_calc_cols)} of the expected {len(all_original_calc_cols)} original columns for calculations.")
    if missing_original_calc_cols:
        print(f"Warning: The following expected columns were NOT found: {missing_original_calc_cols}")
else:
    print("DataFrame not loaded. Skipping column verification.")

Found 92 of the expected 92 original columns for calculations.


## 3. Define Helper Function for Scale Inversion

This function inverts items measured on a Likert scale. It assumes a 1-5 scale (1->5, 2->4, 3->3, 4->2, 5->1). It also handles potential non-numeric values by converting them to NaN.

In [4]:
def invert_likert_scale(series, scale_max=5):
    """
    Inverts values on a Likert scale (e.g., 1 to 5).

    Args:
        series (pd.Series): The pandas Series containing the scale values.
        scale_max (int): The maximum value of the scale (default is 5).

    Returns:
        pd.Series: The Series with inverted scale values. Non-numeric values
                   and values outside the expected range [1, scale_max]
                   will be converted to NaN.
    """
    # Ensure the input is numeric, coercing errors to NaN
    numeric_series = pd.to_numeric(series, errors='coerce')

    # Invert the scale: (Max + 1) - value
    inverted_series = (scale_max + 1) - numeric_series

    # Optional: Add validation to ensure result is within bounds (or handle original values outside 1-5)
    # For now, we assume input is meant to be 1-5, and inversion logic is correct.
    # Values originally outside 1-5 or non-numeric will result in NaN after inversion.

    return inverted_series

# Example usage (optional test)
# test_series = pd.Series([1, 2, 3, 4, 5, 'NA', 6])
# print(invert_likert_scale(test_series))

## 4. Assess Scale Reliability (Cronbach's Alpha) - IMPORTANT STEP

**Crucial:** Before averaging items to create a composite score (like 'Perceived_Autonomy' or 'Burnout_Score'), it's essential to check if the items reliably measure the same underlying construct. Cronbach's Alpha ($\alpha$) is a common measure for this internal consistency.

- **Interpretation:**
  - $\alpha > 0.9$: Excellent
  - $\alpha > 0.8$: Good
  - $\alpha > 0.7$: Acceptable
  - $\alpha > 0.6$: Questionable
  - $\alpha > 0.5$: Poor
  - $\alpha < 0.5$: Unacceptable

**Action:** You should calculate Cronbach's Alpha for *each* set of columns you intend to average. If the alpha is low (e.g., < 0.7 or even < 0.6), averaging them might not be appropriate. You might need to:
  1. Re-evaluate the scale: Are the items truly related?
  2. Remove problematic items: Check item-total correlations.
  3. Use a different modeling approach: Treat items as individual indicators in SEM or use factor scores instead of simple averages.

**Note:** The code below requires the `pingouin` library (`pip install pingouin`). It's commented out as it requires executing the notebook. You *must* run this analysis.

In [5]:
# --- Reliability Analysis Placeholder ---

# Example for 'Perceived Autonomy' scale
# Ensure you have the 'pingouin' library installed: pip install pingouin
# import pingouin as pg # Uncomment

scale_name = 'Perceived Autonomy'
cols_to_check = cols_perceived_autonomy # Use the list defined earlier
items_to_invert = cols_autonomy_to_invert

# --- Prepare data for reliability analysis ---
if 'df' in locals() and not df.empty and all(col in df.columns for col in cols_to_check):
    print(f"\n--- Preparing data for Reliability Analysis: {scale_name} ---")
    # Select the relevant columns
    reliability_df = df[cols_to_check].copy()

    # Convert all columns to numeric, coercing errors
    for col in reliability_df.columns:
        reliability_df[col] = pd.to_numeric(reliability_df[col], errors='coerce')

    # Invert the necessary items *before* calculating alpha
    for col in items_to_invert:
        if col in reliability_df.columns:
            # Assuming a 1-5 scale for inversion
            reliability_df[col] = invert_likert_scale(reliability_df[col], scale_max=5)
            print(f"  Inverted item: {col}")

    # Drop rows with any NaNs *within this scale* for the alpha calculation
    reliability_df.dropna(inplace=True)

    # --- Calculate Cronbach's Alpha (Requires Pingouin) ---
    if not reliability_df.empty and len(reliability_df.columns) > 1:
        print(f"  Calculating Cronbach's Alpha for {len(reliability_df)} complete cases...")
        try:
            # Uncomment the following lines to run the analysis:
            # alpha_results = pg.cronbach_alpha(data=reliability_df)
            # cronbach_alpha = alpha_results[0]
            # print(f"  Cronbach's Alpha for '{scale_name}': {cronbach_alpha:.3f}")
            #
            # # You can also get confidence intervals:
            # # confidence_interval = alpha_results[1]
            # # print(f"  95% Confidence Interval: {confidence_interval}")
            #
            # # Check item-total correlations if alpha is low (optional):
            # # item_stats = pg.read_itemstats(data=reliability_df, cronbach_alpha=cronbach_alpha)
            # # display(item_stats)
            pass # Remove this pass when uncommenting above
        except NameError:
             print("  Skipping Cronbach's Alpha calculation (requires 'pingouin' library).")
        except Exception as e:
            print(f"  Error calculating Cronbach's Alpha for '{scale_name}': {e}")
    elif len(reliability_df.columns) <= 1:
         print("  Skipping Cronbach's Alpha: Scale has only one item.")
    else:
        print("  Skipping Cronbach's Alpha: No complete cases found for this scale after handling NaNs/inversion.")

else:
    print(f"\nSkipping Reliability Analysis for {scale_name}: DataFrame not loaded or columns missing.")

# **TODO:** Repeat this block for ALL other scales you plan to average (e.g., Burnout, Motivations, etc.)


--- Preparing data for Reliability Analysis: Perceived Autonomy ---
  Inverted item: 3. At work, I often feel like I have to follow other people’s commands.
  Inverted item: 4. If I could choose, I would do things at work differently.
  Inverted item: 7. In my job, I feel forced to do things I do not want to do. 
  Calculating Cronbach's Alpha for 2666 complete cases...


## 5. Calculate Composite Variables (Averages)

Now, we calculate the average scores for each composite variable. This involves:
1. Selecting the relevant columns.
2. Ensuring data is numeric.
3. Applying scale inversion where needed using the `invert_likert_scale` function.
4. Calculating the row-wise mean, skipping NaNs (`skipna=True`).

**Assumption:** Based on the reliability analysis (Step 4), we proceed with averaging. If reliability was low for any scale, this step should be reconsidered for that scale.

In [6]:
# --- Function to Calculate Composite Score ---
def calculate_composite_score(df_input, new_col_name, item_cols, invert_cols=[], scale_max=5):
    """
    Calculates a composite score by averaging specified items, handling inversion.

    Args:
        df_input (pd.DataFrame): The input DataFrame.
        new_col_name (str): The name for the new composite score column.
        item_cols (list): List of column names for the items in the scale.
        invert_cols (list, optional): List of columns within item_cols to invert. Defaults to [].
        scale_max (int, optional): Maximum value of the Likert scale. Defaults to 5.

    Returns:
        pd.DataFrame: DataFrame with the new composite score column added.
                      Returns the original DataFrame if columns are missing.
    """
    df_output = df_input.copy()
    existing_item_cols = [col for col in item_cols if col in df_output.columns]
    existing_invert_cols = [col for col in invert_cols if col in existing_item_cols]

    if len(existing_item_cols) != len(item_cols):
        missing = [col for col in item_cols if col not in existing_item_cols]
        print(f"Warning: Cannot calculate '{new_col_name}'. Missing columns: {missing}. Skipping.")
        return df_output # Return original df if columns are missing

    if not existing_item_cols:
         print(f"Warning: No columns found for '{new_col_name}'. Skipping.")
         return df_output

    # Create a temporary DataFrame with only the relevant columns
    temp_df = df_output[existing_item_cols].copy()

    # Convert all columns to numeric
    for col in temp_df.columns:
        temp_df[col] = pd.to_numeric(temp_df[col], errors='coerce')

    # Apply inversion
    for col in existing_invert_cols:
        temp_df[col] = invert_likert_scale(temp_df[col], scale_max=scale_max)

    # Calculate the mean
    df_output[new_col_name] = temp_df.mean(axis=1, skipna=True)
    print(f"Calculated: {new_col_name}")
    return df_output

# --- Apply the function for each composite variable ---
if 'df' in locals() and not df.empty:
    print("\n--- Calculating Composite Variables ---")

    # Work Conditions / Environment
    df = calculate_composite_score(df, 'Academic_Resources', cols_academic_resources)
    # Performance Pressure is a single item, just ensure it's numeric
    if col_performance_pressure in df.columns:
         df['Performance_Pressure'] = pd.to_numeric(df[col_performance_pressure], errors='coerce')
         print(f"Processed single item: Performance_Pressure")
    else:
         print(f"Warning: Column '{col_performance_pressure}' not found for Performance_Pressure.")

    df = calculate_composite_score(df, 'Perceived_Autonomy', cols_perceived_autonomy, cols_autonomy_to_invert)
    df = calculate_composite_score(df, 'Quality_Leadership', cols_quality_leadership)
    df = calculate_composite_score(df, 'Sense_Community', cols_sense_community)
    df = calculate_composite_score(df, 'Job_Satisfaction', cols_job_satisfaction)

    # Work Motivation
    df = calculate_composite_score(df, 'Amotivation', cols_amotivation)
    df = calculate_composite_score(df, 'Extrinsic_Social', cols_extrinsic_social)
    df = calculate_composite_score(df, 'Extrinsic_Material', cols_extrinsic_material)
    df = calculate_composite_score(df, 'Introjected_Motivation', cols_introjected)
    df = calculate_composite_score(df, 'Identified_Motivation', cols_identified)
    df = calculate_composite_score(df, 'Intrinsic_Motivation', cols_intrinsic)

    # Burnout
    df = calculate_composite_score(df, 'Burnout_Score', cols_burnout)

    # Vulnerability / Attitudes (AVEM)
    df = calculate_composite_score(df, 'Subjective_Significance', cols_subjective_significance)
    df = calculate_composite_score(df, 'Professional_Ambition', cols_professional_ambition)
    df = calculate_composite_score(df, 'Tendency_Exertion', cols_tendency_exertion)
    df = calculate_composite_score(df, 'Striving_Perfection', cols_striving_perfection)
    df = calculate_composite_score(df, 'Emotional_Distancing', cols_emotional_distancing, cols_distancing_to_invert)
    df = calculate_composite_score(df, 'Resignation_Tendency', cols_resignation_tendencies)
    df = calculate_composite_score(df, 'Offensive_Coping', cols_offensive_coping)
    df = calculate_composite_score(df, 'Mental_Stability', cols_mental_stability, cols_stability_to_invert)
    df = calculate_composite_score(df, 'Satisfaction_Work', cols_satisfaction_work)
    df = calculate_composite_score(df, 'Satisfaction_Life', cols_satisfaction_life)
    df = calculate_composite_score(df, 'Social_Support', cols_social_support, cols_support_to_invert)

    print("--- Composite Variable Calculation Complete ---")

else:
    print("\nSkipping composite variable calculation as DataFrame is not loaded or empty.")


--- Calculating Composite Variables ---
Calculated: Academic_Resources
Processed single item: Performance_Pressure
Calculated: Perceived_Autonomy
Calculated: Quality_Leadership
Calculated: Sense_Community
Calculated: Job_Satisfaction
Calculated: Amotivation
Calculated: Extrinsic_Social
Calculated: Extrinsic_Material
Calculated: Introjected_Motivation
Calculated: Identified_Motivation
Calculated: Intrinsic_Motivation
Calculated: Burnout_Score
Calculated: Subjective_Significance
Calculated: Professional_Ambition
Calculated: Tendency_Exertion
Calculated: Striving_Perfection
Calculated: Emotional_Distancing
Calculated: Resignation_Tendency
Calculated: Offensive_Coping
Calculated: Mental_Stability
Calculated: Satisfaction_Work
Calculated: Satisfaction_Life
Calculated: Social_Support
--- Composite Variable Calculation Complete ---


## 6. Clean Up: Combine Position, Drop Original Items and Unnecessary Columns

1.  **Combine Position Columns:** Merge country-specific position columns (`CZ_15...`, `AT_15...`) into a single `Current_Position` column.
2.  **Drop Original Items:** Remove the individual survey item columns that were used to create the composite scores.
3.  **Drop Other Unnecessary Columns:** Remove specified columns like 'Unnamed: 4', 'Nationality', duplicate hours column, 'Income CZK'.

In [7]:
if 'df' in locals() and not df.empty:
    print("\n--- Starting Column Cleanup ---")
    initial_cols = set(df.columns)

    # --- 6.1 Combine Position Columns ---
    col_position_cz = 'CZ_15. Your current position at the higher education institution, where you primarily work: '
    col_position_at = 'AT_15. Your current position at the higher education institution, where you primarily work: '
    new_position_col = 'Current_Position'

    # Initialize the new column with NaNs
    df[new_position_col] = np.nan

    # Use combine_first to fill the new column, prioritizing CZ then AT if they exist
    if col_position_cz in df.columns:
        df[new_position_col] = df[new_position_col].combine_first(df[col_position_cz])
        print(f"Combined '{col_position_cz}' into '{new_position_col}'.")
    if col_position_at in df.columns:
        df[new_position_col] = df[new_position_col].combine_first(df[col_position_at])
        print(f"Combined '{col_position_at}' into '{new_position_col}'.")

    # --- 6.2 Define Columns to Drop ---
    # Start with the original calculation columns that actually existed
    columns_to_drop = [col for col in all_original_calc_cols if col in df.columns]
    print(f"Identified {len(columns_to_drop)} original calculation columns to drop.")

    # Add other specific columns to drop
    other_cols_to_drop = [
        'Unnamed: 4',
        '3. Nationality:',
        col_position_cz, # Original CZ position column
        col_position_at, # Original AT position column
        '14.1. Actual average weekly working hours outside higher education (in a typical semester week): .1', # Duplicate hours
        'Income CZK'
    ]

    # Add only those other columns that actually exist in the DataFrame
    for col in other_cols_to_drop:
        if col in df.columns and col not in columns_to_drop:
            columns_to_drop.append(col)

    print(f"Adding other specific columns to drop list: {[col for col in other_cols_to_drop if col in df.columns]}")

    # --- 6.3 Perform Dropping ---
    if columns_to_drop:
        cols_actually_dropped = [col for col in columns_to_drop if col in df.columns]
        df.drop(columns=cols_actually_dropped, inplace=True, errors='ignore') # Use errors='ignore' for safety
        print(f"\nDropped {len(cols_actually_dropped)} columns.")
        # Verify which columns were intended vs dropped if needed
        # dropped_set = set(cols_actually_dropped)
        # intended_set = set(columns_to_drop)
        # if dropped_set != intended_set:
        #     print(f"Warning: Intended to drop {len(intended_set)}, actually dropped {len(dropped_set)}")
        #     print(f"  Columns intended but not dropped (likely didn't exist): {list(intended_set - dropped_set)}")
    else:
        print("\nNo columns identified for dropping.")

    final_cols = set(df.columns)
    print(f"Cleanup complete. Shape after dropping: {df.shape}")
    # print(f"Columns removed: {list(initial_cols - final_cols)}") # Optional: list removed columns

else:
    print("\nSkipping column cleanup as DataFrame is not loaded or empty.")


--- Starting Column Cleanup ---
Combined 'CZ_15. Your current position at the higher education institution, where you primarily work: ' into 'Current_Position'.
Combined 'AT_15. Your current position at the higher education institution, where you primarily work: ' into 'Current_Position'.
Identified 92 original calculation columns to drop.
Adding other specific columns to drop list: ['Unnamed: 4', '3. Nationality:', 'CZ_15. Your current position at the higher education institution, where you primarily work: ', 'AT_15. Your current position at the higher education institution, where you primarily work: ', '14.1. Actual average weekly working hours outside higher education (in a typical semester week): .1', 'Income CZK']

Dropped 98 columns.
Cleanup complete. Shape after dropping: (2708, 59)


## 7. Rename Columns for Clarity and Consistency

Rename columns with long survey questions to shorter, more meaningful names. Add prefixes `WM_` (Work Motivation) and `VB_` (Vulnerability to Burnout / Attitudes) to the respective composite scores for better organization.

In [8]:
if 'df' in locals() and not df.empty:
    print("\n--- Renaming Columns ---")

    rename_dict = {
        # Sociodemographics & Basic Info
        '1. Gender:': 'Gender',
        '2. Age (in years):': 'Age',
        '4. Current marital (partnership) status:': 'Marital_Status',
        '5. Do you currently care for underage children or dependent relatives?': 'Cares_for_Dependents',
        '17. The highest level of education attained:': 'Education_Level',

        # Institution & Role
        '6. The type of higher education insitution where you primarily work:': 'Institution_Type',
        '7. Subject area of the faculty (higher education institution) where you primarily work:': 'Subject_Area',
        '8. Duration of your current employment contract at the higher education institution where you primarily work:': 'Contract_Duration',
        '9. Extent of employment in higher education (in hours/week, aggregated for all higher education institutions where you work):': 'Employment_Hours_HE', # This might be string '40 hours', needs check
        '10. Actual average weekly working hours in higher education (in a typical semester week):': 'Avg_Work_Hours_HE',
        '12. Do you hold a leadership position at a higher education institution?': 'Holds_Leadership_Position',
        '13. How influential are you in helping to shape key academic policies at your institution at the level of department or similar unit?': 'Policy_Influence',
        '16. Please choose the category that best fits your job description:': 'Job_Description_Category', # Likely Non-Academic only
        '18. Total length of your career in Czech higher education in years:': 'Career_Length_HE', # Likely Non-Academic only

        # Other Work
        '14. Do you currently have another (paid) job outside higher education?': 'Has_Other_Job',
        '14.1. Actual average weekly working hours outside higher education (in a typical semester week): ': 'Avg_Work_Hours_Other', # Note potential trailing space

        # Effort Allocation (Percentages) - Keep original names for now, maybe shorten later if needed
        # 'Teaching %': 'Effort_Teaching_Pct',
        # 'Research %': 'Effort_Research_Pct',
        # 'Activities related to externally funded research projects %': 'Effort_External_Projects_Pct',
        # 'Organisational and administrative activities %': 'Effort_Org_Admin_Pct',

        # Effort Allocation (Descriptions - might be specific to Academics)
        '1. Teaching (classroom instruction, preparation of instructional materials and lesson plans, advising students, reading and evaluating student work, examination management, etc.)': 'Effort_Teaching_Desc',
        '2. Research (reading literature, designing and conducting experiments, collecting and analysing data, writing articles or other scientific texts, etc.)': 'Effort_Research_Desc',
        '3. Activities related to externally funded research projects (searching for information on available funding sources, preparation of grant applications and project reports, project management and administration, etc.)': 'Effort_External_Projects_Desc',
        '4. Organisational and administrative activities (organising and attending meetings, dealing with tasks and documents not directly related to teaching, research, or externally funded research projects, etc.)': 'Effort_Org_Admin_Desc',

        # Calculated Effort Comparison
        'Effort (less, more, equal)': 'Effort_Comparison',

        # Income related (Keep as is for now, check types later)
        # 'Income EURO': 'Income_EURO',
        # 'Euro Adj.': 'Income_Euro_Adj',
        # 'Salary/hour': 'Salary_Per_Hour', # Check type, might be object
        # 'Salary effort/hour': 'Salary_Effort_Per_Hour', # Check type

        # Add prefixes to Work Motivation variables
        'Amotivation': 'WM_Amotivation',
        'Extrinsic_Social': 'WM_Extrinsic_Social',
        'Extrinsic_Material': 'WM_Extrinsic_Material',
        'Introjected_Motivation': 'WM_Introjected_Motivation',
        'Identified_Motivation': 'WM_Identified_Motivation',
        'Intrinsic_Motivation': 'WM_Intrinsic_Motivation',

        # Add prefixes to Vulnerability to Burnout / Attitude variables
        'Subjective_Significance': 'VB_Subjective_Significance',
        'Professional_Ambition': 'VB_Professional_Ambition',
        'Tendency_Exertion': 'VB_Tendency_Exertion',
        'Striving_Perfection': 'VB_Striving_Perfection',
        'Emotional_Distancing': 'VB_Emotional_Distancing',
        'Resignation_Tendency': 'VB_Resignation_Tendency',
        'Offensive_Coping': 'VB_Offensive_Coping',
        'Mental_Stability': 'VB_Mental_Stability',
        'Satisfaction_Work': 'VB_Satisfaction_Work',
        'Satisfaction_Life': 'VB_Satisfaction_Life',
        'Social_Support': 'VB_Social_Support'
        # Note: 'Academic_Resources', 'Performance_Pressure', 'Perceived_Autonomy',
        # 'Quality_Leadership', 'Sense_Community', 'Job_Satisfaction', 'Burnout_Score'
        # keep their calculated names without prefixes for now.
    }

    # Apply renaming only for columns that exist in the DataFrame
    current_columns = df.columns
    rename_map_applied = {old: new for old, new in rename_dict.items() if old in current_columns}
    df.rename(columns=rename_map_applied, inplace=True)

    print(f"Applied renaming to {len(rename_map_applied)} columns.")
    print("\n--- Columns after renaming (first 50): ---")
    print(df.columns.tolist()[:50])
    print(f"\nTotal columns remaining: {len(df.columns)}")

else:
    print("\nSkipping renaming as DataFrame is not loaded or empty.")


--- Renaming Columns ---
Applied renaming to 38 columns.

--- Columns after renaming (first 50): ---
['Country', 'Version', 'Gender', 'Age', 'Marital_Status', 'Cares_for_Dependents', 'Institution_Type', 'Subject_Area', 'Contract_Duration', 'Employment_Hours_HE', 'Avg_Work_Hours_HE', 'Effort_Comparison', 'Effort [%]', 'Income EURO', 'Euro Adj.', 'Salary/hour', 'Salary effort/hour', 'Holds_Leadership_Position', 'Policy_Influence', 'Has_Other_Job', 'Avg_Work_Hours_Other', 'Academic/Non-academic', 'Job_Description_Category', 'Education_Level', 'Career_Length_HE', 'Effort_Teaching_Desc', 'Effort_Research_Desc', 'Effort_External_Projects_Desc', 'Effort_Org_Admin_Desc', 'Teaching %', 'Research %', 'Activities related to externally funded research projects %', 'Organisational and administrative activities %', 'Vulnerability', 'Academic_Resources', 'Performance_Pressure', 'Perceived_Autonomy', 'Quality_Leadership', 'Sense_Community', 'Job_Satisfaction', 'WM_Amotivation', 'WM_Extrinsic_Social',

## 8. Handle Missing Values (NaNs) and Outliers

This section implements a structured approach to handle missing data and potential outliers.

**Pipeline:**
1.  **Initial NaN Assessment:** Analyze the extent and pattern of missingness. Using `missingno` library is recommended for visualization (code commented out).
2.  **Define Structural/Acceptable NaNs:** Identify columns where NaNs are expected or acceptable based on survey logic (e.g., questions only asked to academics or non-academics).
3.  **Data Type Conversion:** Ensure columns intended for numeric analysis (like income, hours) are actually numeric type. Convert if possible, flag issues otherwise.
4.  **Outlier Removal (Income):** Remove rows with extreme income values using the Interquartile Range (IQR) method on relevant numeric income columns.
5.  **KNN Imputation:** Impute remaining NaNs in *suitable numeric columns* using K-Nearest Neighbors. Excludes identifiers, categorical, structural NaN columns, and the target variable (Burnout). Uses scaling internally.
6.  **Final NaN Removal:** Drop rows that *still* contain NaNs in columns where they are *not* considered acceptable (i.e., excluding the structural NaN columns).

In [9]:
if 'df' in locals() and not df.empty:
    print(f"\n--- Starting NaN and Outlier Handling ---")
    print(f"Shape before handling: {df.shape}")
    rows_before_handling = len(df)

    # --- 8.1 Initial NaN Assessment (Visualization Recommended) ---
    print("\n--- Initial NaN Counts (Top 20) ---")
    nan_counts = df.isnull().sum()
    print(nan_counts[nan_counts > 0].sort_values(ascending=False).head(20))

    # --- Visualizing Missingness (Optional but Recommended) ---
    # Requires matplotlib and missingno: pip install missingno
    # import missingno as msno
    # import matplotlib.pyplot as plt
    #
    # print("\n--- Missing Data Pattern Matrix ---")
    # msno.matrix(df.sample(500) if len(df) > 500 else df) # Sample if large dataset
    # plt.show()
    #
    # print("\n--- Missing Data Heatmap ---")
    # msno.heatmap(df)
    # plt.show()
    #
    # print("\n--- Missing Data Dendrogram ---")
    # msno.dendrogram(df)
    # plt.show()

    # --- 8.2 Define Structural/Acceptable NaN Columns ---
    # Columns where NaNs might be expected depending on role (Academic/Non-Academic) or survey flow.
    # NaNs in these columns might not necessarily indicate bad data for a given row.
    # **Review and adjust this list based on your deep understanding of the survey logic.**
    structural_nan_cols = [
        # Non-Academic specific (potentially)
        'Job_Description_Category', 'Education_Level', 'Career_Length_HE',
        # Academic specific (potentially - effort descriptions/percentages)
        'Effort_Teaching_Desc', 'Effort_Research_Desc', 'Effort_External_Projects_Desc',
        'Effort_Org_Admin_Desc', 'Teaching %', 'Research %',
        'Activities related to externally funded research projects %',
        'Organisational and administrative activities %',
        # Other job specific
        'Avg_Work_Hours_Other',
        # Add any other columns where NaNs are structurally expected
    ]
    # Ensure these columns actually exist
    structural_nan_cols = [col for col in structural_nan_cols if col in df.columns]
    print(f"\nDefined {len(structural_nan_cols)} structural/acceptable NaN columns: {structural_nan_cols}")


    # --- 8.3 Data Type Conversion (Crucial Before Numeric Operations) ---
    print("\n--- Checking and Converting Data Types ---")
    cols_to_make_numeric = [
        'Age', 'Employment_Hours_HE', 'Avg_Work_Hours_HE', 'Avg_Work_Hours_Other',
        'Career_Length_HE', 'Income EURO', 'Euro Adj.', 'Salary/hour', 'Salary effort/hour',
        'Teaching %', 'Research %', 'Activities related to externally funded research projects %',
        'Organisational and administrative activities %'
        # Add any other columns that should be numeric but might be object/string
    ]
    potential_numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    for col in cols_to_make_numeric:
        if col in df.columns and col not in potential_numeric_cols:
            print(f"Attempting to convert '{col}' to numeric...")
            original_nan_count = df[col].isnull().sum()
            df[col] = pd.to_numeric(df[col], errors='coerce')
            new_nan_count = df[col].isnull().sum()
            if new_nan_count > original_nan_count:
                print(f"  Warning: Conversion of '{col}' introduced {new_nan_count - original_nan_count} new NaNs.")
        elif col not in df.columns:
             print(f"  Skipping conversion for '{col}': Column not found.")

    # Verify types again
    print("\n--- Data types after attempted conversion (Object columns): ---")
    print(df.select_dtypes(include='object').info())


    # --- 8.4 Outlier Removal (Income - IQR Method) ---
    print("\n--- Step 8.4: Removing Income Outliers (IQR) ---")
    cols_for_iqr = ['Income EURO', 'Euro Adj.', 'Salary/hour', 'Salary effort/hour']
    # Use only columns that exist and are now numeric
    cols_for_iqr_numeric = [
        col for col in cols_for_iqr
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    ]

    rows_before_iqr = len(df)
    outlier_indices = pd.Index([])

    if cols_for_iqr_numeric:
        print(f"Applying IQR outlier detection to: {cols_for_iqr_numeric}")
        for col in cols_for_iqr_numeric:
            if df[col].isnull().all() or df[col].nunique() < 2:
                print(f"  Skipping IQR for '{col}': All NaN or single value.")
                continue

            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            if pd.isna(Q1) or pd.isna(Q3) or IQR == 0:
                 print(f"  Skipping IQR for '{col}': Could not calculate valid Q1/Q3/IQR.")
                 continue

            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            col_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].index
            if not col_outliers.empty:
                 print(f"  Found {len(col_outliers)} potential outliers in '{col}' (Bounds: {lower_bound:.2f} - {upper_bound:.2f})")
                 outlier_indices = outlier_indices.union(col_outliers)

        if not outlier_indices.empty:
            df = df.drop(outlier_indices)
            print(f"Removed {len(outlier_indices)} rows identified as outliers based on income columns.")
        else:
            print("No outliers found based on IQR for income columns.")
    else:
        print("No suitable numeric income columns found for IQR outlier removal.")

    rows_after_iqr = len(df)
    print(f"Shape after IQR: {df.shape}")


    # --- 8.5 KNN Imputation ---
    print("\n--- Step 8.5: Imputing Missing Values (KNN) ---")
    # Identify columns suitable for KNN imputation: MUST be numeric.
    # Exclude identifiers, categorical vars, the target (Burnout), and structural NaNs.
    cols_to_exclude_from_knn = list(set(structural_nan_cols + [
        'Country', 'Version', # Identifiers/Metadata if they exist
        'Gender', 'Marital_Status', 'Cares_for_Dependents', 'Institution_Type',
        'Subject_Area', 'Contract_Duration', 'Effort_Comparison',
        'Holds_Leadership_Position', 'Policy_Influence', 'Has_Other_Job',
        'Academic/Non-academic', 'Current_Position', # Categorical/Object
        'Burnout_Score', # Exclude target variable
        # Add any other ID or purely categorical columns
    ]))
    cols_to_exclude_from_knn = [col for col in cols_to_exclude_from_knn if col in df.columns]

    # Select numeric columns NOT in the exclusion list
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    cols_for_knn = [col for col in numeric_cols if col not in cols_to_exclude_from_knn]

    if not cols_for_knn:
        print("No suitable numeric columns found for KNN imputation. Skipping.")
    else:
        print(f"Selected {len(cols_for_knn)} numeric columns for KNN imputation.")
        # print(f"Columns for KNN: {cols_for_knn}") # Uncomment to list columns

        df_knn_subset = df[cols_for_knn].copy()
        nans_before_knn = df_knn_subset.isnull().sum().sum()
        print(f"Total NaNs to impute in selected columns: {nans_before_knn}")

        if nans_before_knn > 0:
            try:
                # Scale -> Impute -> Inverse Scale
                scaler = StandardScaler()
                # Fit scaler only on non-NaN data? Or fit on all and let imputer handle NaNs?
                # Standard practice: Fit scaler, transform, impute, inverse_transform
                scaled_data = scaler.fit_transform(df_knn_subset)

                imputer = KNNImputer(n_neighbors=5)
                imputed_scaled_data = imputer.fit_transform(scaled_data) # Imputes based on scaled data

                # Inverse transform to get back original scale
                imputed_data = scaler.inverse_transform(imputed_scaled_data)

                # Create a temporary DataFrame with imputed values
                df_imputed_subset = pd.DataFrame(imputed_data, columns=cols_for_knn, index=df.index) # Use original index!

                # Update the original DataFrame
                df.update(df_imputed_subset)

                nans_after_knn = df[cols_for_knn].isnull().sum().sum()
                print(f"KNN Imputation complete. NaNs remaining in imputed columns: {nans_after_knn}")
                if nans_after_knn > 0:
                    print("  Warning: Some NaNs might remain after KNN (e.g., if all neighbors were NaN).")

            except Exception as e:
                print(f"  ERROR during KNN Imputation: {e}. Skipping imputation.")
        else:
            print("No NaNs found in the selected columns for KNN imputation.")

    print(f"Shape after KNN attempt: {df.shape}")


    # --- 8.6 Final NaN Removal (Selective Drop) ---
    print("\n--- Step 8.6: Removing Rows with Remaining Non-Acceptable NaNs ---")
    # Identify columns where NaNs are NOT acceptable (all columns MINUS structural ones)
    cols_to_check_final_nans = [col for col in df.columns if col not in structural_nan_cols]

    if not cols_to_check_final_nans:
        print("Warning: No columns identified for final NaN check (all might be structural?). Skipping drop.")
    else:
        initial_rows = len(df)
        # Drop rows where ANY NaN exists in the non-structural columns
        df.dropna(subset=cols_to_check_final_nans, inplace=True)
        final_rows = len(df)
        print(f"Removed {initial_rows - final_rows} rows containing NaNs in non-structural columns.")

    # --- Final Check ---
    print("\n--- NaN/Outlier Handling Complete ---")
    print(f"Final DataFrame shape: {df.shape}")
    rows_removed_total = rows_before_handling - len(df)
    print(f"Total rows removed during handling: {rows_removed_total} ({rows_removed_total / rows_before_handling:.2%} of original)")

    print("\nFinal remaining NaNs per column (should ideally only be in structural columns):")
    final_nans = df.isnull().sum()
    print(final_nans[final_nans > 0])

else:
    print("\nSkipping NaN/Outlier handling as DataFrame is not loaded or empty.")


--- Starting NaN and Outlier Handling ---
Shape before handling: (2708, 59)

--- Initial NaN Counts (Top 20) ---
Education_Level                                                2176
Career_Length_HE                                               2172
Job_Description_Category                                       2123
Avg_Work_Hours_Other                                           1855
Activities related to externally funded research projects %    1310
Research %                                                     1249
Organisational and administrative activities %                 1246
Teaching %                                                     1237
Effort_External_Projects_Desc                                   741
Effort_Research_Desc                                            656
Effort_Org_Admin_Desc                                           653
Effort_Teaching_Desc                                            635
Academic_Resources                                              593
Sa

## 9. Save Processed Data

Save the cleaned and processed DataFrame to a new file. Using formats like Parquet or Feather is often more efficient for intermediate data storage than CSV or Excel, but CSV/Excel are provided for compatibility.

In [10]:
if 'df' in locals() and not df.empty:
    # Define output directory and filenames
    output_dir = './' # Save in the current directory, or specify a path e.g., '/content/drive/MyDrive/labour well being/Processed Data/'
    csv_output_filename = output_dir + 'labour_wellbeing_processed_v1.csv'
    excel_output_filename = output_dir + 'labour_wellbeing_processed_v1.xlsx'
    parquet_output_filename = output_dir + 'labour_wellbeing_processed_v1.parquet' # Recommended

    # Create directory if it doesn't exist (optional)
    # import os
    # os.makedirs(output_dir, exist_ok=True)

    print(f"\n--- Saving Processed Data ---")
    try:
        # Save to CSV (using utf-8-sig for better Excel compatibility with special chars)
        df.to_csv(csv_output_filename, index=False, encoding='utf-8-sig')
        print(f"DataFrame saved to CSV: {csv_output_filename}")

        # Save to Excel
        # df.to_excel(excel_output_filename, index=False, engine='openpyxl') # Requires openpyxl
        # print(f"DataFrame saved to Excel: {excel_output_filename}")

        # Save to Parquet (Recommended)
        df.to_parquet(parquet_output_filename, index=False) # Requires pyarrow or fastparquet
        print(f"DataFrame saved to Parquet: {parquet_output_filename}")

    except Exception as e:
        print(f"Error saving data: {e}")

elif 'df' in locals() and df.empty:
     print("\nSkipping saving: DataFrame is empty after processing.")
else:
    print("\nSkipping saving: DataFrame not loaded.")


--- Saving Processed Data ---
DataFrame saved to CSV: ./labour_wellbeing_processed_v1.csv
Error saving data: ("Could not convert '0,5' with type str: tried to convert to double", 'Conversion failed for column Effort_Org_Admin_Desc with type object')


--- End of Data Preparation ---